In [ ]:
import gc

import pandas as pd
import xgboost as xgb
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from default_risk.scripts.auxiliars_for_modeling import apply_cyclical_encoding
import lightgbm as lgb
import optuna
import mlflow
import numpy as np
import yaml
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import re
import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import get_pipeline

from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from optuna_integration.mlflow import MLflowCallback


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
#mlflow.xgboost.autolog(log_models=False)



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_test-processed.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = application_train_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
model= xgb.XGBClassifier(**hiperparams)
run_cv_tracked_mlflow(model,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application + instalament + credit card + cash balance")

#freeing memory
del merged_df
gc.collect()



In [ ]:
#for the second one  we gonna analize the gains from the aggregation of bureau
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_test-processed.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

model= xgb.XGBClassifier(**hiperparams)

run_cv_tracked_mlflow(model,hiperparams,cv,X,Y,experiment_name,"application_train + bureau + bureau_balance")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#now bureau parent (main with feature engineering + Bureau with feature engineering + Bureau_balance)
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "toxic_baseline.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)



#importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")

#X= clean_importance_zero_and_negative_pfi(importance_df,X)
#importance_permutation_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_bureau_parent.csv")
importance_df= pd.read_csv(cfg.ARTIFACTS_DIR / "external_importance.csv")
#criteria = creating_criteria(importance_df,importance_permutation_df)

#importance2 = pd.read_csv(cfg.ARTIFACTS_DIR / "second filter.csv") #
#X= X.drop(columns=["bureau_balance_is_delincuency_sum_loan_1","bureau_has_bureau_balance_data_loan_1","ext_source_1_is_missing"]) 

X= clean_noise_from_feature_importance(importance_df,X,0.004)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"external_parent_co_sample_clean")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#Now let's try the external historial with target encoding
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

model= xgb.XGBClassifier(**hiperparams)
categorical_features=  ["organization_type","occupation_type","name_income_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)





run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"external_parent_target_encoding")



#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()



X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

feature_raper= pd.read_csv(cfg.ARTIFACTS_DIR / "final_importance.csv")

X= clean_noise_from_feature_importance(feature_raper,X)

merged_df= merged_df.drop(columns=["flag_email"])

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")

#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "pipeline_baseline.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)


#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")



merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()





X,Y = prepare_columns(merged_df)

X = cast_object_into_categoricals(X)

#importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_max_cols_internal.csv")




#X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00010)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"max_rows_final_model")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

features_from_internal_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "internal_best_result.csv")
features_from_external_historial= pd.read_csv(cfg.ARTIFACTS_DIR / "external_best_result.csv")

internal_list=  features_from_internal_historial["feature_name"].to_list()
external_list=  features_from_external_historial["feature_name"].to_list()
features_names = list(set(internal_list + external_list))

X,Y = prepare_columns(merged_df)

X= X[features_names]

X= X.drop(columns= ["amt_down_payment_sum"])



X = cast_object_into_categoricals(X)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"final_model_from_convination_of_best_results")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

In [ ]:
#main + prev_app + bureau + installments with LGBM
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

par = {
'n_estimators': 4693,
'learning_rate': 0.007553343326775532,
'num_leaves': 37,
'min_child_samples': 220,
'min_child_weight': 0.0028908609938903796,
"max_depth" :-1,           
'subsample': 0.8389118020044092,    
'subsample_freq': 3,   
'colsample_bytree': 0.5616610992553818,
'reg_alpha': 2.24385145804699e-07,
'reg_lambda': 7.747405452353306,
'min_split_gain': 0.786374413940256, 
"random_state" : 42,
'cat_smooth': 7.237675739015409,
'cat_l2': 75.34738775153713,
"n_jobs" : 6,
"objective" : 'binary',
"force_col_wise": True,
"importance_type" : "gain"
}

par2 = {
'n_estimators': 4854,
'learning_rate': 0.00587633931427308,
'num_leaves': 40,
'min_child_samples': 30,
'min_child_weight': 0.0010075425502278205,
"max_depth" :-1,           
'subsample': 0.8379685394911657,    
'subsample_freq': 5,   
'colsample_bytree': 0.5007314038467754,
'reg_alpha': 5.874001031590365,
'reg_lambda': 2.473060711126495,
'min_split_gain': 0.41384599131751537, 
"random_state" : 42,
'cat_smooth': 91.59649623757437,
'cat_l2': 28.173404262475664,
"n_jobs" : 6,
"objective" : 'binary',
"force_col_wise": True,
"importance_type" : "gain"
}

best_params_optuna = {'target_enc_smooth': 2.5821099004254604, 'n_estimators': 4693, 'learning_rate': 0.007553343326775532, 'num_leaves': 37, 'min_child_samples': 220, 'min_child_weight': 0.0028908609938903796, 'subsample': 0.8389118020044092, 'subsample_freq': 3, 'colsample_bytree': 0.5616610992553818, 'reg_alpha': 2.24385145804699e-07, 'reg_lambda': 7.747405452353306, 'min_split_gain': 0.786374413940256, 'cat_smooth': 7.237675739015409, 'cat_l2': 75.34738775153713}

Mejores_Hiperparámetros= {'target_enc_smooth': 78.0016624983871, 'n_estimators': 4854, 'learning_rate': 0.005876339314273085, 'num_leaves': 40, 'min_child_samples': 77, 'min_child_weight': 0.0010075425502278205, 'subsample': 0.8379685394911657, 'subsample_freq': 5, 'colsample_bytree': 0.5007314038467754, 'reg_alpha': 5.874001031590365, 'reg_lambda': 2.473060711126495, 'min_split_gain': 0.41384599131751537, 'cat_smooth': 91.59649623757437, 'cat_l2': 28.173404262475664}
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

model_lgbm = lgb.LGBMClassifier(**par2)


categorical_features= ["organization_type","occupation_type","wallsmaterial_mode"] # #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1", "organization_type" "organization_type",


pipeline= get_pipeline(2.5821099004254604,categorical_features,model_lgbm)





#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df,santize_text=True)







X["kui_external"] =  np.where(X["days_employed"] != 0, X["active_amt_credit_sum_debt_active_sum"] / ((X["days_employed"] * -1) * X["amt_income_total"]), 0)

X["ratio_of_payment_external_debt"] = np.where(X["active_amt_credit_sum_debt_active_sum"] != 0,X["active_amt_credit_sum_active_sum"] / X["active_amt_credit_sum_debt_active_sum"],np.nan)

X["antique_annuity_vs_actual_annuity"] = np.where(X["amt_annuity"],X["amt_annuity_median"] / X["amt_annuity"],np.nan)

X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)



X['random_noise'] = np.random.normal(0, 1, len(X))



X = cast_object_into_categoricals(X)


model=xgb.XGBClassifier(**hiperparams)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00001)

snd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_first_cut.csv")

X = clean_importance_zero_and_negative_pfi(snd_filter,X,0.00001)

trd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_third_cut.csv")

X = clean_importance_zero_and_negative_pfi(trd_filter,X,0.00007)

last_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_for_pipeline.csv")

X = clean_importance_zero_and_negative_pfi(last_filter,X,0.00005)

slow_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_last_cut.csv")

X= pd.get_dummies(X, columns= ["education_type"])

#X = clean_importance_zero_and_negative_pfi(slow_filter,X)

run_cv_tracked_mlflow(pipeline,par,cv,X,Y,experiment_name,"lightgbm_last_cut")

#auc_score_OOF=  0.781

#freeing memory
del merged_df
gc.collect()

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()




In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)

#X= clean_colineality(X,list_to_delete)


model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
 #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

X = cast_object_into_categoricals(X)



#X= X.drop(columns=cols_to_drop)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_monster_final_model.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,-0.00001)

second_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "second.csv")

X = clean_importance_zero_and_negative_pfi(second_filter,X,-0.00001)

third_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "third.csv")

X = clean_importance_zero_and_negative_pfi(third_filter,X,-0.00001)

fourth_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "fourth.csv")

X = clean_importance_zero_and_negative_pfi(fourth_filter,X,0.00001)



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"monster_without_co_lineality",enable_feature_permutation=False)

#auc_score_OOF=  0.781

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target_encoding.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)



#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)

#X= clean_colineality(X,list_to_delete)




model= xgb.XGBClassifier(**hiperparams)
categorical_features= ["organization_type","occupation_type"] #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
 #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1",
pipeline= get_pipeline(50,categorical_features,model)


X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)

X = cast_object_into_categoricals(X)

cols_to_drop= ["name_income_type"]

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_monster_final_model.csv")


X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,-0.00001)

second_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "second.csv")

X = clean_importance_zero_and_negative_pfi(second_filter,X,-0.00001)

third_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "third.csv")

X = clean_importance_zero_and_negative_pfi(third_filter,X,-0.00001)

fourth_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "fourth.csv")

X = clean_importance_zero_and_negative_pfi(fourth_filter,X,0.00001)

X= X.drop(columns=cols_to_drop)






#five_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_fine_pruned_final_model.csv")

#X = clean_importance_zero_and_negative_pfi(five_filter,X,0.00009)

run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"fine_pruned_final_model")


In [ ]:
def objective(trial, X, Y, categorical_features):


    target_enc_smooth = trial.suggest_float("target_enc_smooth", 1.0, 100.0, log=True)

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 5000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
        "max_bin": trial.suggest_categorical("max_bin", [63, 127, 255]),

        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7), # Activa el uso de subsample
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),

        "cat_smooth": trial.suggest_float("cat_smooth", 1.0, 100.0, log=True),
        "cat_l2": trial.suggest_float("cat_l2", 1e-2, 100.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 20.0),

        "max_depth": -1, 
        "random_state": 42,
        "n_jobs": 14,
        "objective": 'binary',       
        "force_col_wise": True,
        "importance_type": "gain"
    }

    model = lgb.LGBMClassifier(**params)


    pipeline = get_pipeline(target_enc_smooth, categorical_features, model)
    
    auc_scores = cross_val_score(
        pipeline, 
        X, 
        Y, 
        cv=5, 
        scoring="roc_auc", 
        n_jobs=1
    )
    
    return np.mean(auc_scores)


def run_optimization(X_train, Y_train, cat_features):
    study = optuna.create_study(
        study_name="lightgbm_tuning",
        direction="maximize" 
    )
    
    study.optimize(
        lambda trial: objective(trial, X_train, Y_train, cat_features), 
        n_trials=300
    )
    
    return study


    

In [ ]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)


application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "main_target.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)




categorical_features= ["organization_type","occupation_type","wallsmaterial_mode"] # #organization_type,"name_goods_category_prev_1","name_cash_loan_purpose_prev_1", "organization_type" "organization_type",








#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "internal_parent_max_cols.parquet")




merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df,santize_text=True)






#X["external_ratio_debt_income"]  =  X["bureau_amt_credit_sum_loan_1"] / X["amt_income_total"]







#X["ratio_credit_active_external"] = np.where(X["amt_income_total"],X["active_amt_credit_sum_active_sum"] / X["amt_income_total"],np.nan)


#X["external_anuity_active_vs_internal_annuity_historial"] = np.where(X["amt_annuity"] != 0, X["active_amt_annuity_active_mean"] / X["amt_annuity_median"] ,np.nan)

#X["external_anuity_closed_vs_internal_annuity_historial"] = np.where((X["amt_annuity"] != 0), X["active_amt_annuity_active_mean"] / X["amt_annuity_median"] ,np.nan)

#X["balance_income_ratio"] = np.where(X["amt_income_total"],X["active_amt_credit_sum_active_sum"] / X["amt_income_total"],np.nan)

X["kui_external"] =  np.where(X["days_employed"] != 0, X["active_amt_credit_sum_debt_active_sum"] / ((X["days_employed"] * -1) * X["amt_income_total"]), 0)

X["ratio_of_payment_external_debt"] = np.where(X["active_amt_credit_sum_debt_active_sum"] != 0,X["active_amt_credit_sum_active_sum"] / X["active_amt_credit_sum_debt_active_sum"],np.nan)

X["antique_annuity_vs_actual_annuity"] = np.where(X["amt_annuity"],X["amt_annuity_median"] / X["amt_annuity"],np.nan)

X["instalment_income_ratio"] = np.where(X["amt_income_total"],X["instalments_amt_instalment_sum_sum"] / X["amt_income_total"],np.nan)



X['random_noise'] = np.random.normal(0, 1, len(X))



X = cast_object_into_categoricals(X)


model=xgb.XGBClassifier(**hiperparams)

importance_pfi_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm.csv")

X = clean_importance_zero_and_negative_pfi(importance_pfi_df,X,0.00001)

snd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_first_cut.csv")

X = clean_importance_zero_and_negative_pfi(snd_filter,X,0.00001)

trd_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_with_third_cut.csv")

X = clean_importance_zero_and_negative_pfi(trd_filter,X,0.00007)

last_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_for_pipeline.csv")

X = clean_importance_zero_and_negative_pfi(last_filter,X,0.00005)

slow_filter= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_lightgbm_last_cut.csv")

X= pd.get_dummies(X, columns= ["education_type"])






study = run_optimization(X,Y,categorical_features)
print(f"Mejor AUC alcanzado: {study.best_value}")
print(f"Mejores Hiperparámetros: {study.best_params}")